# Module 2: Product Analytics — Walkthrough

This notebook documents the full process behind Module 2 (RFM segmentation, cohort/retention, churn flag + drivers) — including the dead ends and why decisions were made, not just the final output. Scoring/aggregation logic lives in SQL views under `warehouse/`; the automated pipeline scripts are under `analytics/product/*.py`. This notebook re-runs the same queries and narrates the reasoning.

Reference: `PROJECT_PLAN.md` Section 6.

In [ ]:
import os, sys

# Step 1: mount Drive (Colab only) and cd into the project folder.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

PROJECT_PATH = "/content/drive/MyDrive/UNT OneDrive Backup/Backup folder/Backup folder/Projects/retail-analytics-experiments"
assert os.path.isdir(PROJECT_PATH), f"Project folder not found at: {PROJECT_PATH}"
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)
print("Step 1 OK - cwd:", os.getcwd())

In [ ]:
from dotenv import dotenv_values

# Step 2: load .env manually (no ambiguous path-searching) and verify
# every required key is present and non-empty BEFORE attempting to connect.
env_path = os.path.join(PROJECT_PATH, ".env")
assert os.path.exists(env_path), f".env not found at: {env_path}"

env_values = dotenv_values(env_path)
required_keys = [
    "SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD",
    "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA",
]
missing = [k for k in required_keys if not env_values.get(k)]
if missing:
    raise ValueError(f".env is missing or has empty values for: {missing}")

# override=True equivalent: force these into os.environ even if a stale
# value from an earlier cell run in this same kernel session is present.
for k, v in env_values.items():
    if v:
        os.environ[k] = v

print("Step 2 OK - .env loaded, keys present:", list(env_values.keys()))

# Step 3: test the Snowflake connection directly, in isolation, before
# running any actual analysis query. If this cell fails, the problem is
# purely connection/credentials -- nothing to do with the notebook's
# import setup or any analysis code below.
import snowflake.connector

test_conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema=os.environ["SNOWFLAKE_SCHEMA"],
    role=os.environ.get("SNOWFLAKE_ROLE"),
)
print("Step 3 OK - Snowflake connection successful:", test_conn)
test_conn.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from analytics.snowflake_utils import query_df

print("Step 4 OK - imports ready")

## Part 1 — RFM Segmentation

**Approach:** Recency / Frequency / Monetary are quintile-scored (1-5) using SQL `NTILE(5)` window functions in `warehouse/vw_rfm_scores.sql`, on top of `vw_customer_summary`. Quintiles are computed independently per dimension, then combined into a segment label.

**Why SQL for scoring, not pandas?** The plan calls for SQL window functions specifically (Section 6.1) — it's the more natural place to do it since the data already lives in Snowflake, and it keeps the heavy aggregation server-side rather than pulling 180K raw rows into the notebook.

In [ ]:
rfm = query_df("SELECT * FROM vw_rfm_scores")
rfm.head()

In [ ]:
segment_counts = rfm["RFM_SEGMENT"].value_counts()
segment_counts

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=segment_counts.index, y=segment_counts.values)
plt.title("Customer count by RFM segment")
plt.ylabel("Customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Observation:** no customers landed in "Champions" (top quintile on all three of R, F, M). This isn't a bug — it's a known characteristic of scoring R/F/M as *independent* quintiles: nothing forces the top-recency, top-frequency, and top-monetary groups to overlap. In this dataset they simply don't intersect. Worth calling out explicitly rather than silently treating it as missing data.

## Part 2 — Cohort & Retention Analysis

**Approach:** cohort = month of a customer's first order (`first_order_date`, derived back in `etl/clean.py`). For each cohort, `period_number` = months between the cohort month and any later order month. Retention % = (distinct customers active in period N) / (cohort size at period 0). Built in `warehouse/vw_cohort_orders.sql`.

In [ ]:
cohort_orders = query_df("SELECT * FROM vw_cohort_orders")

cohort_sizes = (
    cohort_orders[cohort_orders["PERIOD_NUMBER"] == 0]
    .groupby("COHORT_MONTH")["CUSTOMER_ID"]
    .nunique()
)
cohort_sizes

**Observation:** cohort sizes jump sharply from low-hundreds to 2,000+ starting around October 2017. That's a sudden, implausible jump in real new-customer acquisition — most likely an artifact of how this dataset was assembled/sampled near its end date, not genuine growth. Flagging this here so it isn't misread as a real acquisition spike in the writeup.

In [ ]:
cohort_counts = (
    cohort_orders.groupby(["COHORT_MONTH", "PERIOD_NUMBER"])["CUSTOMER_ID"]
    .nunique()
    .reset_index()
    .pivot(index="COHORT_MONTH", columns="PERIOD_NUMBER", values="CUSTOMER_ID")
)
retention = cohort_counts.divide(cohort_sizes, axis=0)

plt.figure(figsize=(14, 8))
sns.heatmap(retention, annot=True, fmt=".0%", cmap="Blues")
plt.title("Customer retention by cohort month")
plt.xlabel("Months since first order")
plt.ylabel("Cohort month")
plt.tight_layout()
plt.show()

**Finding:** retention stabilizes at roughly 11-17% from month 1 onward, across nearly every cohort, with no strong upward or downward trend over time (aside from the acquisition-spike cohorts noted above, which don't have enough elapsed months yet to compare).

## Part 3 — Churn Flag & Drivers

**Definition:** churn = no order in the last 90 days relative to the dataset's max order date (`warehouse/vw_churn_flag.sql`). This is a fixed-window flag, not a model — straightforward and matches the project's "standard techniques only" scope (Section 6.3).

In [ ]:
churn = query_df("SELECT * FROM vw_churn_flag")

print(f"Overall churn rate: {churn['IS_CHURNED'].mean():.1%}")
churn.groupby("CUSTOMER_SEGMENT")["IS_CHURNED"].mean()

### Attempt 1 — logistic regression with order count as a feature

The plan (Section 6.3) suggests an *optional* logistic regression using RFM features + segment + region to explain churn drivers. First attempt: throw in order count, total sales, avg order value, and segment, and fit a plain MLE logit. Recency itself is deliberately excluded as a feature since it's literally what defines the churn label (would be circular).

In [ ]:
features_v1 = churn[["ORDER_COUNT", "TOTAL_SALES", "AVG_ORDER_VALUE", "CUSTOMER_SEGMENT"]].copy()
features_v1 = pd.get_dummies(features_v1, columns=["CUSTOMER_SEGMENT"], drop_first=True)
features_v1 = sm.add_constant(features_v1.astype(float))
target = churn["IS_CHURNED"].astype(int)

model_v1 = sm.Logit(target, features_v1).fit(disp=False)
print(model_v1.summary())

**What went wrong:** `converged: False`, absurd coefficients (`ORDER_COUNT` ~18, `const` ~-20), and a "possibly complete quasi-separation" warning. Checking the raw relationship explains why:

In [ ]:
order_count_bucket = churn["ORDER_COUNT"].clip(upper=3).map({1: "1 order", 2: "2 orders", 3: "3+ orders"})
churn.groupby(order_count_bucket)["IS_CHURNED"].mean()

Every customer with 2+ orders is churned (100%) in this dataset. That's a structural artifact: repeat purchases happen in a tight early burst per customer, so by the dataset's end date almost all multi-order customers necessarily have >90 days since their last order. `ORDER_COUNT` therefore near-perfectly separates the two classes — logistic regression can't fit finite coefficients under that condition. This is a genuine, useful finding about the dataset (worth reporting on its own), but it has to come out of the model as a feature.

### Attempt 2 — L1-regularized fit (tried to keep order count in)

Tried `fit_regularized(method="l1", alpha=0.1)` as the standard workaround for separation. It "converged" but the regularization just pinned `ORDER_COUNT`'s coefficient to ~0 and the other coefficients had enormous standard errors (hundreds) — not a usable result, since the regularized covariance estimate isn't reliable for inference here. Dropped this approach rather than tune the penalty further; reporting `ORDER_COUNT`'s effect descriptively (above) is more honest and clearer anyway.

### Attempt 3 — drop order count, keep both sales features

Removed `ORDER_COUNT`, kept `TOTAL_SALES` and `AVG_ORDER_VALUE` together. This *also* failed to converge, with both coefficients getting huge opposite-signed values and enormous standard errors (~5,159) that almost canceled out.

In [ ]:
features_v3 = churn[["TOTAL_SALES", "AVG_ORDER_VALUE", "CUSTOMER_SEGMENT"]].copy()
features_v3 = pd.get_dummies(features_v3, columns=["CUSTOMER_SEGMENT"], drop_first=True)
features_v3 = sm.add_constant(features_v3.astype(float))

model_v3 = sm.Logit(target, features_v3).fit(disp=False)
print(model_v3.summary())

**Root cause:** `TOTAL_SALES` and `AVG_ORDER_VALUE` are near-perfectly collinear here — they're *identical* for the ~70% of customers with exactly one order, since avg = total when count = 1. Two collinear features in the same model means the solver can't separate their individual effects, hence the blown-up, opposite-signed, mutually-canceling coefficients.

### Final model — avg order value + segment only

Dropped `TOTAL_SALES` (redundant with `AVG_ORDER_VALUE` given the collinearity above) and `ORDER_COUNT` (causes separation, reported descriptively instead). This converges cleanly.

In [ ]:
features_final = churn[["AVG_ORDER_VALUE", "CUSTOMER_SEGMENT"]].copy()
features_final = pd.get_dummies(features_final, columns=["CUSTOMER_SEGMENT"], drop_first=True)
features_final = sm.add_constant(features_final.astype(float))

model_final = sm.Logit(target, features_final).fit(disp=False)
print(model_final.summary())

## Module 2 conclusions

- **RFM:** segments are clean and usable, but "Champions" is empty by construction of independent-quintile scoring — note this caveat in any writeup.
- **Cohort retention:** stabilizes at ~11-17% from month 1 onward across nearly all cohorts; cohort sizes spike implausibly from Oct 2017 onward, which looks like a dataset-assembly artifact rather than real acquisition growth.
- **Churn:** purchase frequency is the dominant driver, but it's so dominant (100% churn at 2+ orders) that it can't be used as a regression feature — better reported as a direct rate-by-bucket finding. Once that's removed, avg order value has a small but statistically significant positive association with churn (higher one-off spend → slightly higher churn likelihood); customer segment has no significant effect.
- **Methodological lesson:** when a logistic regression won't converge, check for (a) quasi-separation from a too-predictive feature, and (b) collinearity between features that are mathematically related (here, avg = total ÷ count) — both showed up in this one model.